# Entrenamiento y comparación de modelos (v1.1.0)

Usa `ft_engineering()` para obtener los datos ya transformados y entrena varios clasificadores.

**Clase de interés:** `0` = NO paga a tiempo. Por eso precisión, recall y F1 se calculan con `pos_label=0`.

> Requiere `ft_engineering.py`, `cargar_datos.py` y `Base_de_datos.xlsx` en la misma carpeta que este notebook.
>
> Este notebook también se **importa** desde `model_evaluation.py`: las celdas de definición (funciones) se ejecutan al importar, y la celda final de ejecución solo corre cuando se abre el notebook directamente.

## 1. Librerías

In [42]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, make_scorer,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from ft_engineering import ft_engineering  # datos limpios, divididos y transformados

# XGBoost es opcional: si no está instalado, se entrenan los demás modelos
try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None
    print("xgboost no está instalado: se entrenan los demás modelos.")

RANDOM_STATE = 1234  # semilla fija para resultados reproducibles

# Métricas que se reportan (model_evaluation.py las usa para tablas y gráficos)
METRICAS = ["accuracy", "precision", "recall", "f1_score", "roc_auc",
            "casosNoPagoAtiempo"]

## 2. Funciones reutilizables

- `score_no_pago`: puntaje continuo de "no paga" (para el ROC AUC).
- `summarize_classification`: métricas sobre la clase 0. El ROC AUC usa el puntaje continuo; con clases duras subestima la capacidad de discriminar.

In [43]:
def score_no_pago(model, X) -> np.ndarray:
    """Puntaje continuo de 'no paga' (mayor = más riesgo), para el ROC AUC."""
    # Modelos con probabilidades: tomamos la columna de la clase 0
    if hasattr(model, "predict_proba"):
        clases = list(model.classes_)
        return model.predict_proba(X)[:, clases.index(0)]
    # LinearSVC no tiene probabilidades: una decisión alta significa clase 1
    # (paga), así que invertimos el signo para que "mayor = más riesgo"
    return -model.decision_function(X)


def summarize_classification(y_true, y_pred, y_score=None) -> dict:
    """Métricas sobre la clase 0 (no paga a tiempo)."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # Si no hay puntaje continuo, el AUC se calcula con las clases duras
    roc_in = y_score if y_score is not None else (y_pred == 0).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        # pos_label=0: medimos qué tan bien detectamos a quienes NO pagan
        "precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "roc_auc": roc_auc_score((y_true == 0).astype(int), roc_in),
        "casosNoPagoAtiempo": int(np.count_nonzero(y_pred == 0)),
    }

## 3. Modelos candidatos

Todos manejan el desbalance de clases (~5% no paga). En XGBoost, la clase positiva es la 1 (la mayoritaria), por eso `scale_pos_weight = n_neg / n_pos` la reduce.

In [44]:
def get_models(y_train) -> dict:
    """Modelos candidatos, todos con manejo del desbalance de clases."""
    n_neg, n_pos = (y_train == 0).sum(), (y_train == 1).sum()
    models = {
        # Los modelos lineales necesitan variables en la misma escala
        "logistic": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(solver="liblinear", class_weight="balanced")),
        ]),
        "svc": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LinearSVC(C=1.0, max_iter=5000, dual=False, class_weight="balanced")),
        ]),
        # Árbol y bosque: profundidad limitada para frenar el sobreajuste
        "decision_tree": DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=10, class_weight="balanced",
            random_state=RANDOM_STATE),
        "random_forest": RandomForestClassifier(
            n_estimators=150, max_depth=7, min_samples_leaf=5, max_features="sqrt",
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    }
    if XGBClassifier is not None:
        models["xgboost"] = XGBClassifier(
            eval_metric="logloss", n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=n_neg / n_pos,  # compensa el desbalance
            random_state=RANDOM_STATE)
    return models

## 4. `build_model`

Entrena un clasificador y devuelve métricas en train y test, más el recall de la clase 0 en validación cruzada estratificada (sobre train).

In [45]:
def build_model(classifier, data: dict, cv_splits: int = 5) -> dict:
    """Entrena un clasificador y devuelve métricas train/test y CV (recall clase 0)."""
    X_train, X_test = data["X_train"], data["X_test"]
    y_train, y_test = data["y_train"], data["y_test"]

    classifier.fit(X_train, y_train)

    # Métricas en train y en test (la brecha entre ambas delata el sobreajuste)
    resumen = {}
    for nombre, X, y in (("train", X_train, y_train), ("test", X_test, y_test)):
        resumen[nombre] = summarize_classification(
            y, classifier.predict(X), score_no_pago(classifier, X))

    # Validación cruzada solo sobre train: estima la estabilidad del recall
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    cv_recall = cross_val_score(classifier, X_train, y_train, cv=cv,
                                scoring=make_scorer(recall_score, pos_label=0))
    resumen["cv_recall_no_pago"] = (cv_recall.mean(), cv_recall.std())
    return resumen

## 5. Selección del mejor modelo y entrenamiento completo

**Criterio de selección:** ranking promedio de **recall (clase 0)** y **ROC AUC** en test.

Usar solo el recall no es suficiente: un modelo que marcara a *todos* los clientes como "no paga" tendría recall 1,0 y sería inútil. El ROC AUC corrige eso porque mide qué tan bien separa a los dos grupos.

`ft_engineering()` excluye `puntaje` por defecto (separa perfectamente a pagadores y no pagadores, lo que apunta a fuga de información). Para incluirla: `train_all(excluir_features=[])`.

In [46]:
def select_best(results_df: pd.DataFrame) -> str:
    """Mejor modelo por ranking promedio de recall (clase 0) y ROC AUC en test."""
    test = results_df[results_df["Data Set"] == "test"]
    tabla = test.pivot(index="Model", columns="Metric", values="Score")
    # rank(ascending=False): el mejor valor recibe el puesto 1
    ranking = tabla[["recall", "roc_auc"]].rank(ascending=False).mean(axis=1)
    return ranking.sort_values().index[0]


def train_all(**kwargs) -> dict:
    """Entrena todos los modelos y devuelve lo que necesita model_evaluation.py.

    kwargs se pasan a ft_engineering (p. ej. excluir_features=[]).
    """
    # 1) Datos limpios, divididos (estratificado) y transformados
    data = ft_engineering(**kwargs)

    # 2) Entrenar y evaluar cada modelo
    models = get_models(data["y_train"])
    resultados, filas = {}, []
    for nombre, clf in models.items():
        resultados[nombre] = build_model(clf, data)
        # Pasamos a formato largo (Model, Data Set, Metric, Score): es el
        # formato que usan las tablas y gráficos de model_evaluation.py
        for conjunto in ("train", "test"):
            for metrica, valor in resultados[nombre][conjunto].items():
                filas.append({"Model": nombre, "Data Set": conjunto,
                              "Metric": metrica, "Score": valor})
    results_df = pd.DataFrame(filas)

    # 3) Elegir el ganador
    best = select_best(results_df)

    return {
        "models": models,
        "results_df": results_df,
        "best": best,
        "data": data,
        "resultados": resultados,  # incluye el recall en validación cruzada
    }


def guardar(res: dict, carpeta: str = ".") -> None:
    """Guarda el mejor modelo, el preprocesador ajustado y la tabla de resultados."""
    os.makedirs(carpeta, exist_ok=True)  # crea la carpeta si no existe
    best = res["best"]
    joblib.dump(res["models"][best], f"{carpeta}/modelo_{best}.joblib")
    joblib.dump(res["data"]["preprocessor"], f"{carpeta}/preprocessor.joblib")
    res["results_df"].to_csv(f"{carpeta}/resultados_modelos.csv", index=False)

## 6. Ejecución

Esta celda **solo corre al abrir el notebook directamente**. Cuando `model_evaluation.py` lo importa, el guard `__name__ == "__main__"` evita que se vuelva a entrenar todo.

In [47]:
if __name__ == "__main__":
    res = train_all()

    tabla = res["results_df"].pivot_table(
        index=["Model", "Data Set"], columns="Metric", values="Score")[METRICAS]
    display(tabla.round(3))

    print("Recall CV (clase 0 = no paga), media ± desvío:")
    for m, r in res["resultados"].items():
        print(f"  {m}: {r['cv_recall_no_pago'][0]:.3f} ± {r['cv_recall_no_pago'][1]:.3f}")

    print("\nMejor modelo:", res["best"])
    guardar(res, carpeta="../salidas")  # raíz del proyecto/salidas

Metric                  accuracy  precision  recall  f1_score  roc_auc  \
Model         Data Set                                                   
decision_tree test         0.618      0.063   0.510     0.112    0.612   
              train        0.632      0.086   0.699     0.153    0.738   
logistic      test         0.655      0.082   0.618     0.145    0.684   
              train        0.653      0.080   0.599     0.141    0.675   
random_forest test         0.806      0.103   0.402     0.164    0.691   
              train        0.844      0.212   0.844     0.339    0.932   
svc           test         0.655      0.081   0.608     0.143    0.684   
              train        0.655      0.080   0.599     0.142    0.674   
xgboost       test         0.895      0.127   0.206     0.157    0.651   
              train        0.961      0.547   1.000     0.707    0.999   

Metric                  casosNoPagoAtiempo  
Model         Data Set                      
decision_tree test                   825.0  
              train                 3333.0  
logistic      test                   766.0  
              train                 3064.0  
random_forest test                   397.0  
              train                 1626.0  
svc           test                   764.0  
              train                 3051.0  
xgboost       test                   165.0  
              train                  748.0

Recall CV (clase 0 = no paga), media ± desvío:
  logistic: 0.584 ± 0.063
  svc: 0.584 ± 0.061
  decision_tree: 0.470 ± 0.061
  random_forest: 0.359 ± 0.049
  xgboost: 0.225 ± 0.023

Mejor modelo: logistic
